# §37: **K5fT2** and **K5dfiT2** = K5f and K5dfi retrained on T2, read zero-shot on whole DoTA-CAP (dev + eval = 1,129)

Rule: `core/docs/final/KIM_RUN.md` **§37**, written before any number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **PRELIMINARY (n = 1, s2024); printed, decides nothing; K5f (Nexar) stays the reference and the main model.** DADA-2000 is not read: T2 is cut from it, so it is in-domain here (§13 T6).

| row | what | here |
|---|---|---|
| **K5fT2** | K5T2's command (§13) on the §31 code: config == K5T2's | **train + read** (control) |
| **K5dfiT2** | K5fT2 + `dvs.theta=0.7`, `dvs.theta_ego=0.85`, `kim.seam_mask=true`, `loss.dvs_anchor_mode=ignore` | **train + read** |
| K5T2 | §13 run (pre-§31 labels) | checkpoint read only: + its DoTA-CAP-eval read, to pool |
| K5f, K5dfi | Nexar-trained, §35 whole DoTA-CAP reads | reads only |
| LaGoVAD | released checkpoint, §35 read | reads only |

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, deps, preflight (K5T2 run + dev read, T2 inputs, bakes, §35 reads) | no | 2-5 min |
| 4 | pytest on the touched code -- **raises if red** | no | 2-3 min |
| 5 | the DoTA-CAP-eval bake with T2 statistics (found, or rebuilt on CPU) | no | 0-10 min |
| 6 | train K5fT2, K5dfiT2 (each about as long as K5T2); J10, config diff to K5T2, `mul_mil` floor | **yes** | 2 x K5T2 |
| 7 | DoTA-CAP-dev + eval (`--final`, marker) for the 2 new rows, eval for K5T2; pool -> whole DoTA-CAP | **yes** | 3 x 5-10 min |
| 8 | `zs_arm_table` (control K5fT2) -> **paste `REPORTS/zs_arm_table_t2_fixed_s2024.md` to Claude** | no | 2-5 min |

Everything new goes to `Thesis-Final/outputs/kim_t2_fixed/`. A re-run skips what is on Drive. **Watch the log: past 2 s/step, stop and paste it.** The eval half is never printed alone.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip (DoTA_s1_ncc)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: T2 data, T2 A2 bakes, KNN, V2-S caches
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5T2, §35 reads; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 37. K5fT2' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §37 -- re-upload kat-vad/'
assert (REPO / 'core/tools/dota_cap_pool.py').is_file(), 'no dota_cap_pool -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/ (training now would repeat K5T2)'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5       = constants.KIM_MAIN_ARM                       # "K5": names run / eval sub-folders only
K5T2     = constants.KIM_T2_ROW                         # §13, pre-§31
K5FT2, K5DFIT2 = constants.KIM_T2_FIXED_ROW, constants.KIM_T2_FIXED_DVS_IGNORE_ROW
NEW_ROWS = (K5FT2, K5DFIT2)
K5F, K5DFI, LAG = constants.KIM_REFERENCE_ARM, constants.KIM_DVS_IGNORE_ROW, constants.ZS_LAGOVAD_ARM
SEED     = constants.KIM_SEED                           # 2024
ENC      = constants.V5_ENCODER                         # V2-S
OFF      = constants.V2_OFF
T2NAME   = constants.DADA_ORIGIN_DATASET
EPOCHS   = constants.KIM_T2_EPOCHS                      # 20
CAP_DEV, CAP_EVAL = constants.V2_SPLIT_DOTA_CAP_DEV, constants.V2_SPLIT_DOTA_CAP_EVAL
CAP_ALL  = constants.DOTA_CAP_ALL_DIRNAME
STRIDE   = constants.V2_E1_STRIDE_BC                    # protocol B: s1[::3]
N_DEV, N_EVAL = constants.DOTA_CAP_DEV_CLIPS, constants.DOTA_CAP_EVAL_CLIPS

OUT_F    = constants.OUTPUT_ROOT
V2_CACHE = Path(DRIVE_V2) / 'cache'
A2       = V2_CACHE / 'v2' / 'A2_S'
# Training: K5T2's (§13)
T2_DATA  = Path(DRIVE_V2) / 'data' / f'{T2NAME}_v2'
T2_KNN   = V2_CACHE / 'knn' / f'{T2NAME}_v2' / constants.KNN_CACHE_FILENAME
A2_T2    = A2 / T2NAME                                  # T2 statistics
K5T2_ROOT = OUT_F / constants.KIM_T2_OUTPUT_DIRNAME
K5T2_RUN = K5T2_ROOT / K5 / f's{SEED}'
# DoTA-CAP inputs with T2 statistics
CAP_BAKE = {CAP_DEV: A2 / 'DoTA_CAP_s3_from_s1',                    # E3's (K5T2's dev read)
            CAP_EVAL: A2 / 'DoTA_CAP_eval_s3_from_s1'}              # v2 Final's
CAP_EVAL_FALLBACK = Path(DRIVE_F) / 'cache' / 'v2' / 'A2_S' / 'DoTA_CAP_eval_s3_from_s1_t2'
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
CAP_V2S_CANDIDATES = [Path(r) / 'cache' / 'video' / ENC / f'{constants.DOTA_CAP_DATASET}_s1_squash'
                      for r in (DRIVE, DRIVE_V2, DRIVE_F)]
# Outputs: one folder per row (K5T2's eval read and pool also land here, never in kim_t2/)
ROOT     = OUT_F / constants.KIM_T2_FIXED_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/k5_t2_fixed')
LOCAL.mkdir(parents=True, exist_ok=True)
CAP35    = OUT_F / 'kim_cap_eval'                       # §35's whole DoTA-CAP reads


def run_dir(row):
    return (K5T2_RUN if row == K5T2 else ROOT / row / K5 / f's{SEED}')


def read_dir(row, split):
    '''protocol-B read of ``split`` for a T2 row; K5T2's dev read is its §13 one.'''
    if row == K5T2 and split == CAP_DEV:
        return K5T2_ROOT / 'eval' / 'dota' / K5
    return ROOT / row / 'eval' / split / K5


TABLE_READS = {K5FT2: read_dir(K5FT2, CAP_ALL), K5DFIT2: read_dir(K5DFIT2, CAP_ALL), K5T2: read_dir(K5T2, CAP_ALL),
               K5F: CAP35 / K5F / 'eval' / CAP_ALL / K5, K5DFI: CAP35 / K5DFI / 'eval' / CAP_ALL / K5,
               LAG: CAP35 / LAG / 'eval' / CAP_ALL / 'B'}
INPUTS = (('T2 data', T2_DATA), ('T2 KNN', T2_KNN), ('T2 A2 bake', A2_T2 / constants.V2_INPUT_MANIFEST_FILENAME),
          ('K5T2 checkpoint', K5T2_RUN / 'checkpoint_last.pt'), ('K5T2 verified', K5T2_RUN / 'verified.json'),
          ('K5T2 config', K5T2_RUN / 'config.yaml'), ('K5T2 metrics', K5T2_RUN / 'metrics.jsonl'),
          ('K5T2 dev read', read_dir(K5T2, CAP_DEV) / 'protocol_b_readout.json'),
          ('CAP-dev bake (T2 stats)', CAP_BAKE[CAP_DEV] / constants.V2_INPUT_MANIFEST_FILENAME), ('DoTA s1', DOTA_S1),
          ('DoTA metadata', DOTA / 'metadata_val.json'),
          *[(f'{r} §35 read', TABLE_READS[r] / 'protocol_b_readout.json') for r in (K5F, K5DFI, LAG)])
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | runs / reads -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 6-7 need one'
df -h /content | tail -1

In [ ]:
import torch
import yaml

from core.config import Config                  # noqa: E402
from core.data.v2_splits import load_split      # noqa: E402
from core.tools.v5_stage1 import _flat          # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def stage(src, dst, ids=None):
    '''A Drive file or dir (only ``ids`` + json/npz manifests if given) to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    if ids is None:
        if not dst.exists():
            tmp = dst.with_name(dst.name + '.partial')
            shutil.rmtree(tmp, ignore_errors=True)
            shutil.copytree(src, tmp)
            os.replace(tmp, dst)
    else:
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(d, steps):
    rec, ckpt = d / 'verified.json', d / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


# Preflight on small files only (C39)
t2_cfg = _flat(yaml.safe_load((K5T2_RUN / 'config.yaml').read_text()))
assert t2_cfg['kim.enabled'] and t2_cfg['data.dataset'] == T2NAME and t2_cfg['dvs.theta'] == 1.0, 'K5T2_RUN is not K5T2'
STEPS = json.loads((K5T2_RUN / 'verified.json').read_text())['step']             # 1,740
assert STEPS == constants.NEXAR_STEP_BUDGET and t2_cfg['train.num_epochs'] == EPOCHS, (STEPS, t2_cfg['train.num_epochs'])
assert is_verified(K5T2_RUN, STEPS) and metrics_rows(K5T2_RUN / 'metrics.jsonl')[-1]['global_step'] == STEPS, 'K5T2: J10'
T2_SHA = manifest(A2_T2)['train_ids_sha1']
assert (manifest(A2_T2)['crn'], manifest(A2_T2)['motion']) == (OFF, ENC), manifest(A2_T2)
CAP_IDS = {CAP_DEV: sorted(load_split(CAP_DEV)), CAP_EVAL: sorted(load_split(CAP_EVAL, final=True))}   # ids only (§37)
assert (len(CAP_IDS[CAP_DEV]), len(CAP_IDS[CAP_EVAL])) == (N_DEV, N_EVAL)


def bake_ok(d, split):
    if not (d / constants.V2_INPUT_MANIFEST_FILENAME).exists():
        return False
    m = manifest(d)
    return ((m['crn'], m['motion'], m['train_ids_sha1']) == (OFF, ENC, T2_SHA)
            and not [v for v in CAP_IDS[split] if not (d / f'{v}.npy').exists()])


assert bake_ok(CAP_BAKE[CAP_DEV], CAP_DEV), f'CAP-dev bake is not T2 statistics on all {N_DEV} clips: {manifest(CAP_BAKE[CAP_DEV])}'
miss = [v for v in CAP_IDS[CAP_DEV] + CAP_IDS[CAP_EVAL] if not (DOTA_S1 / f'{v}.npy').exists()]
assert not miss, f'{len(miss)} DoTA-CAP clips have no CLIP s1 rows: {miss[:5]}'
print(f'preflight OK: K5T2 step {STEPS} ({EPOCHS} epochs) | T2 sha {T2_SHA[:12]} | DoTA-CAP {N_DEV} + {N_EVAL}')

## 4. Tests on the touched code (CPU) -- **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_dota_cap_pool.py',
     'core/tests/test_baseline_losses.py', 'core/tests/test_dvs_dataset.py', 'core/tests/test_kim.py'])

## 5. The DoTA-CAP-eval bake with T2 statistics: v2 Final's, or rebuilt here (CPU, `build_v2_inputs apply`, never `fit`)

In [ ]:
EVAL_BAKE = next((d for d in (CAP_BAKE[CAP_EVAL], CAP_EVAL_FALLBACK) if bake_ok(d, CAP_EVAL)), None)
if EVAL_BAKE is None:
    cap_u = next((d for d in CAP_V2S_CANDIDATES if d.is_dir()
                  and not [v for v in CAP_IDS[CAP_EVAL] if not (d / f'{v}.npy').exists()]), None)
    assert cap_u is not None, (f'no T2-statistics CAP-eval bake and no V2-S rows of the {N_EVAL} eval clips under '
                               f'{[str(p) for p in CAP_V2S_CANDIDATES]} -- paste this to Claude, do not improvise')
    local = LOCAL / 'bake_cap_eval'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply',
         '--stats-dir', stage(A2_T2, LOCAL / 'stage' / 't2_stats', []),
         '--clip-dir', stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS[CAP_EVAL]),
         '--video-dir', stage(cap_u, LOCAL / 'stage' / 'cap_v2s', CAP_IDS[CAP_EVAL]),
         '--ids-file', write_ids(LOCAL / 'ids' / f'{CAP_EVAL}.txt', CAP_IDS[CAP_EVAL]), '--stride', STRIDE,
         '--out-dir', local], log=LOCAL / 'bake.log')
    publish(local, CAP_EVAL_FALLBACK)
    flush_drive()
    assert bake_ok(CAP_EVAL_FALLBACK, CAP_EVAL), f'rebuilt bake fails its check: {manifest(CAP_EVAL_FALLBACK)}'
    EVAL_BAKE = CAP_EVAL_FALLBACK
CAP_IN = {CAP_DEV: CAP_BAKE[CAP_DEV], CAP_EVAL: EVAL_BAKE}
print('DoTA-CAP inputs (T2 statistics):', {s: str(d) for s, d in CAP_IN.items()})

## 6. Train K5fT2 and K5dfiT2 (§37 R1)

K5T2's `--set` list (§13); K5dfiT2 replaces four keys. Checks: J10, config diff to K5T2 (K5fT2: none; K5dfiT2: exactly the four keys), last-epoch `mul_mil` > the §31 floor (K5T2 ended at 0.0026). **Watch the log: past 2 s/step, stop and paste it.**

In [ ]:
K5_SETS = ['kim.enabled=true', 'kim.prototypes=true', 'kim.edge_mask=true', 'dvs.theta=1.0', 'dvs.theta_ego=1.0']
BASE = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
        f'data.dataset={T2NAME}', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
        'kip.enabled=false', 'loss.dvs_anchor_mode=span', 'data.is_egocentric=false',
        f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}',
        f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}', *K5_SETS]
ARMS = {K5FT2: {},
        K5DFIT2: {'dvs.theta': '0.7', 'dvs.theta_ego': '0.85', 'kim.seam_mask': 'true',
                  'loss.dvs_anchor_mode': constants.DVS_ANCHOR_MODE_IGNORE}}
DEFAULTS = _flat(Config().to_dict())


def sets_for(row):
    '''K5T2's --set list with the row's overrides replacing the same keys (never a second copy of a key).'''
    over = ARMS[row]
    return [*[s for s in BASE if s.split('=', 1)[0] not in over], *[f'{k}={v}' for k, v in over.items()]]


def config_diff(base, got):
    '''Keys that differ; a key added after ``base`` trained may sit at its default.'''
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k):
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out


def check_run(row):
    d = run_dir(row)
    rows = metrics_rows(d / 'metrics.jsonl')
    assert ckpt_step(d / 'checkpoint_last.pt') == rows[-1]['global_step'] == STEPS, f'{row}: J10'
    got = _flat(yaml.safe_load((d / 'config.yaml').read_text()))
    diff = config_diff(t2_cfg, got)
    assert set(diff) == set(ARMS[row]), f'{row} must differ from K5T2 in exactly {sorted(ARMS[row])}: {diff}'
    last = [r for r in rows if r['epoch'] == rows[-1]['epoch']]
    mul_last = sum(r['mul_mil'] for r in last) / len(last)
    assert mul_last > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{row}: last-epoch mul_mil {mul_last:.4f} -- pre-§31 code'
    ref = metrics_rows(K5T2_RUN / 'metrics.jsonl')[0]
    print(f'{row}: config diff to K5T2 {diff} | last-epoch mul_mil {mul_last:.4f} > {constants.KIM_FIXED_MUL_MIL_FLOOR}')
    print(f"  step 1 (printed): mil {rows[0]['mil']:.6f} (K5T2 {ref['mil']:.6f}) | mul_mil {rows[0]['mul_mil']:.6f} "
          f"(K5T2 {ref['mul_mil']:.6f})")


def train_row(row):
    d = run_dir(row)
    if is_verified(d, STEPS):
        print(f'{row} s{SEED}: verified on Drive -- not retrained')
        check_run(row)
        return
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / row
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in sets_for(row) for x in ('--set', s)],
         '--data-dir', stage(T2_DATA, LOCAL / 'stage' / 't2_data'), '--clip-dir', stage(A2_T2, LOCAL / 'stage' / 't2_a2'),
         '--knn-cache', stage(T2_KNN, LOCAL / 'stage' / 'knn.npz'), '--output-dir', local, *resume],
        log=LOCAL / f'train_{row}.log')
    last = metrics_rows(local / 'metrics.jsonl')[-1]
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{row}: J10 locally {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, d / name)
    flush_drive()
    assert ckpt_step(d / 'checkpoint_last.pt') == STEPS, f'{row}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': (d / 'checkpoint_last.pt').stat().st_size}))
    publish(local / 'verified.json', d / 'verified.json')
    print(f'{row}: trained in {(time.time() - t0) / 60:.0f} min | mil {last.get("mil")} | kim_mil {last.get("kim_mil")}')
    check_run(row)


for row in NEW_ROWS:
    print(row, 'overrides:', ARMS[row])
    train_row(row)

## 7. DoTA-CAP reads: dev + eval for K5fT2 / K5dfiT2, eval for K5T2 (`--final`, §37), then pool each row -> whole DoTA-CAP

In [ ]:
from core.tools import dota_cap_pool  # noqa: E402

MARK = ROOT / 'OPENED.json'
if not MARK.exists():
    ROOT.mkdir(parents=True, exist_ok=True)
    MARK.write_text(json.dumps({'opened_utc': time.strftime('%Y-%m-%dT%H:%MZ', time.gmtime()), 'split': CAP_EVAL,
                                'rows': [*NEW_ROWS, K5T2], 'seed': SEED, 'authorized': 'user, 2026-11-21',
                                'rule': 'KIM_RUN.md §37: printed, decides nothing',
                                'checkpoints': {r: str(run_dir(r) / 'checkpoint_last.pt') for r in (*NEW_ROWS, K5T2)}}, indent=2))
L_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS[CAP_DEV] + CAP_IDS[CAP_EVAL])
L_IN = {s: stage(CAP_IN[s], LOCAL / 'stage' / f'cap_{s}', CAP_IDS[s]) for s in CAP_IN}
L_LAB = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')

for row in (*NEW_ROWS, K5T2):
    ckpt = run_dir(row) / 'checkpoint_last.pt'
    for split in (CAP_DEV, CAP_EVAL):
        out = read_dir(row, split)
        if not (out / 'protocol_b_readout.json').exists():
            local = LOCAL / 'eval' / row / split
            shutil.rmtree(local, ignore_errors=True)
            final = ['--final'] if split == CAP_EVAL else []
            run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ckpt, *final,
                 '--input-dir', L_IN[split], '--s1-dir', L_S1, '--metadata', DOTA / 'metadata_val.json',
                 '--split-file', DOTA / 'val_split.txt', '--data-dir', L_LAB, '--split', split, '--stride', STRIDE,
                 '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{row}_{split}.log')
            publish(local, out)
        r = json.loads((out / 'protocol_b_readout.json').read_text())
        assert r['split'] == split and r['clips'] == len(CAP_IDS[split]) and r['global_steps'] == {f's{SEED}': STEPS}, r
        assert (r['arm']['crn'], r['arm']['motion']) == (OFF, ENC), r['arm']
    out = read_dir(row, CAP_ALL)
    if not (out / 'protocol_b_readout.json').exists():
        local = LOCAL / 'eval' / row / CAP_ALL
        shutil.rmtree(local, ignore_errors=True)
        dota_cap_pool.write_pooled(read_dir(row, CAP_DEV), read_dir(row, CAP_EVAL), local, (N_DEV, N_EVAL))
        publish(local, out)
    print(f'{row}: whole DoTA-CAP pooled ({json.loads((out / "protocol_b_readout.json").read_text())["clips"]} clips)')
flush_drive()

## 8. `zs_arm_table` on whole DoTA-CAP (1,129), control **K5fT2** -> paste `REPORTS/zs_arm_table_t2_fixed_s2024.md` to Claude

Rows K5fT2, K5dfiT2, K5T2 (pre-§31), K5f and K5dfi (Nexar-trained), LaGoVAD (public ckpt). Δ = row − K5fT2: flip the sign of `K5f - K5fT2` for "T2 minus Nexar". Printed only; it decides nothing.

In [ ]:
from core.tools import zs_arm_table as zat  # noqa: E402

NAME = f't2_fixed_s{SEED}'
local = LOCAL / 'report'
old_pool, old_name = zat.POOLING[zat.DOTA], zat.SET_NAME[zat.DOTA]
zat.POOLING[zat.DOTA] = f'{N_DEV + N_EVAL} clips, protocol B; micro / AP after per-clip min-max'
zat.SET_NAME[zat.DOTA] = 'DoTA-CAP (dev+eval)'
try:
    reads = [a for row, d in TABLE_READS.items() for a in ('--read', zat.DOTA, row, str(d))]
    zat.main([*reads, '--control', K5FT2, '--seed', str(SEED), '--name', NAME, '--out-dir', str(local)])
finally:
    zat.POOLING[zat.DOTA], zat.SET_NAME[zat.DOTA] = old_pool, old_name
md_path = local / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.md'
md_path.write_text(md_path.read_text().replace('DoTA-CAP-dev and DADA-2000 after', 'DoTA-CAP (dev+eval) after'))
publish(local, REPORTS)
print((REPORTS / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f'Drive flushed -- paste REPORTS/{constants.ZS_ARM_TABLE_STEM}_{NAME}.md to Claude')